# Notebook 07 — Multi-Feature Batch GD with Feature Scaling (Advertising Dataset)

**Session:** Linear Regression — L1, L2, and Huber Losses (Level 2)  
**Module:** 02 · Calculus and Linear Algebra

## Goal

Train the same multi-feature linear model on the Advertising dataset using **full-batch
gradient descent** instead of SGD.  One weight update is performed per epoch after
accumulating gradients over all $N = 200$ samples.

**Learning objectives:**
- Compare per-epoch loss curves between batch GD and SGD.
- Note that batch GD requires more epochs than SGD but produces smoother loss curves.
- Understand the importance of correctly averaging accumulated gradients.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

## 1  Load and Scale Data

In [ ]:
def get_column(data: list, index: int) -> list:
    return [row[index] for row in data]


def mean_range_scale(
    *feature_lists: list,
) -> tuple[list[list], tuple[float, float, float]]:
    combined = [v for col in feature_lists for v in col]
    max_v = max(combined)
    min_v = min(combined)
    mean_v = sum(combined) / len(combined)
    spread = max_v - min_v
    scaled = [[(x - mean_v) / spread for x in col] for col in feature_lists]
    return scaled, (mean_v, max_v, min_v)


data = np.genfromtxt("../data/advertising.csv", delimiter=",", skip_header=1).tolist()
N = len(data)

tv_data = get_column(data, 0)
radio_data = get_column(data, 1)
newspaper_data = get_column(data, 2)
sales_data = get_column(data, 3)

(tv_scaled, radio_scaled, newspaper_scaled), (mean_v, max_v, min_v) = mean_range_scale(
    tv_data, radio_data, newspaper_data
)

print(f"N = {N}  |  scaling stats: mean={mean_v:.2f}, max={max_v:.2f}, min={min_v:.2f}")

## 2  Model Functions

In [ ]:
def predict(
    x1: float,
    x2: float,
    x3: float,
    w1: float,
    w2: float,
    w3: float,
    b: float,
) -> float:
    return w1 * x1 + w2 * x2 + w3 * x3 + b


def compute_gradient_wi(xi: float, y: float, y_hat: float) -> float:
    return 2 * xi * (y_hat - y)


def compute_gradient_b(y: float, y_hat: float) -> float:
    return 2 * (y_hat - y)


def initialise_params(seed: int = 42) -> tuple[float, float, float, float]:
    random.seed(seed)
    w1 = random.gauss(mu=0.0, sigma=1.0)
    w2 = random.gauss(mu=0.0, sigma=1.0)
    w3 = random.gauss(mu=0.0, sigma=1.0)
    return w1, w2, w3, 0.0

## 3  Batch Gradient Descent Loop

> **Note:** `epoch_max = 25 000` matches the original source notebook.
> Reduce it if you want faster iteration (the model converges well before 25 000 epochs).

In [ ]:
w1, w2, w3, b = initialise_params()
lr: float = 0.01
epoch_max: int = 25_000
epoch_losses: list[float] = []

for _epoch in range(epoch_max):
    loss_total = 0.0
    dw1_total = dw2_total = dw3_total = db_total = 0.0

    for i in range(N):
        x1 = tv_scaled[i]
        x2 = radio_scaled[i]
        x3 = newspaper_scaled[i]
        y = sales_data[i]

        y_hat = predict(x1, x2, x3, w1, w2, w3, b)
        loss_total += (y_hat - y) ** 2

        dw1_total += compute_gradient_wi(x1, y, y_hat)
        dw2_total += compute_gradient_wi(x2, y, y_hat)
        dw3_total += compute_gradient_wi(x3, y, y_hat)
        db_total += compute_gradient_b(y, y_hat)

    # averaged gradient update (once per epoch)
    epoch_losses.append(loss_total / N)
    w1 -= lr * dw1_total / N
    w2 -= lr * dw2_total / N
    w3 -= lr * dw3_total / N
    b -= lr * db_total / N

print(f"Final  w1={w1:.4f}, w2={w2:.4f}, w3={w3:.4f}, b={b:.4f}")

## 4  Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epoch_losses)
axes[0].set_title("Batch GD — MSE Loss per Epoch (all)")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Mean MSE")

axes[1].plot(epoch_losses[-100:])
axes[1].set_title("Batch GD — MSE Loss (last 100 epochs)")
axes[1].set_xlabel("Epoch")

plt.tight_layout()
plt.show()

## 5  Inference

In [ ]:
def scale_test_point(
    tv: float,
    radio: float,
    newspaper: float,
    mean_v: float,
    max_v: float,
    min_v: float,
) -> tuple[float, float, float]:
    spread = max_v - min_v
    return (
        (tv - mean_v) / spread,
        (radio - mean_v) / spread,
        (newspaper - mean_v) / spread,
    )


tv_t, radio_t, newspaper_t = scale_test_point(230.1, 37.8, 69.2, mean_v, max_v, min_v)
print(
    f"Predicted sales (230.1, 37.8, 69.2): {predict(tv_t, radio_t, newspaper_t, w1, w2, w3, b):.2f}"
)

tv_t, radio_t, newspaper_t = scale_test_point(19.2, 35.9, 51.3, mean_v, max_v, min_v)
print(
    f"Predicted sales (19.2,  35.9, 51.3) : {predict(tv_t, radio_t, newspaper_t, w1, w2, w3, b):.2f}"
)